In [1]:
import os
import pandas as pd
import librosa
import soundfile as sf
import numpy as np
import random
import shutil
import logging
from sklearn.model_selection import train_test_split

# -----------------------------------------------------------------------------
# Configuration & Constants
# -----------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

ROOT = r'D:\IMADS\CoffeeGrinder'
SUBDS = 'STWINonGrinder'
METADATA_FILE = os.path.join(ROOT, '2025 dataset planning.xlsx')
BASE_DIR = os.path.join(ROOT, SUBDS)
SEGMENTS_DIR = os.path.join(BASE_DIR, 'segments')
DURATION_IN_SECONDS = 5
SAMPLING_RATE_MIC = 16000
TARGET_SNR_DB = -4

# Mapping for renaming/dropping columns based on subset
COLUMN_MAP = {
    'STWINonGrinder': ('ID acq machine', 'ID acq table'),
    'STWINonTable': ('ID acq table', 'ID acq machine'),
    'STWINonTable_onlyExp': ('ID acq table', 'ID acq machine'),
}

# -----------------------------------------------------------------------------
# Domain Shift Parameters
# -----------------------------------------------------------------------------
SOURCE_DOMAIN_GRINDS = [0, 100, 200]
BACKGROUND_DOMAIN_MAPPING = {
    'A': 'source',
    'B': 'source',
    'C': 'source',
    'D': 'source',
    'E': 'target',
    'F': 'source',
    'Z': 'target',
}

<h2>Step 1: Read and preprocess the metadata config.</h2>

In [2]:
# -----------------------------------------------------------------------------
# Helper Functions - Excel 
# -----------------------------------------------------------------------------
def read_excel_sheet(filename, sheet_name):
    """Reads an Excel sheet and returns a DataFrame."""
    try:
        df = pd.read_excel(filename, sheet_name=sheet_name)
        return df
    except Exception as e:
        logging.error(f"Error reading sheet '{sheet_name}' from file '{filename}': {e}")
        raise

def process_config_dataframe(df, subset):
    """
    Process the config DataFrame:
    - Retrieve relevant columns,
    - Rename a key column to 'ID acq'
    """
    df = df.loc[:, ['Bkg noise', 'Type', 'Model', 'pos', 'grinder precision (um)','ID acq table', 'ID acq machine', 'Anomaly type']]
    
    # Rename and drop columns depending on subset
    rename_col, drop_col = COLUMN_MAP.get(subset, (None, None))
    if drop_col and drop_col in df.columns:
        df = df.drop(columns=[drop_col])
    if rename_col and rename_col in df.columns:
        df.rename(columns={rename_col: 'ID acq'}, inplace=True)
    else:
        logging.warning(f"Column '{rename_col}' not found; skipping rename.")
            
    return df

In [4]:
# Step 1: Read and preprocess the metadata config.
config_df = read_excel_sheet(METADATA_FILE, 'Config')
config_df = process_config_dataframe(config_df, SUBDS)

config_df

,Bkg noise,Type,Model,pos,grinder precision (um),ID acq,Anomaly type
0,\,Normal,White,A,0,20250404_15_43_05,NaN
1,\,Normal,White,A,0,20250404_15_41_56,NaN
2,\,Normal,White,A,0,20250408_16_29_45,NaN
3,\,Normal,White,A,0,20250408_16_28_39,NaN
4,\,Normal,White,A,0,20250403_14_52_31,NaN
...,...,...,...,...,...,...,...
125,E,Not working,\,B,\,20250408_15_02_04,NaN
126,E,Not working,\,C,\,20250408_15_31_58,NaN
127,F,Not working,\,A,\,20250411_15_49_56,NaN
128,F,Not working,\,B,\,20250411_16_06_29,NaN


<h2>Step 2: Build the primary dataset and background DataFrames.</h2>

In [5]:
# -----------------------------------------------------------------------------
# Helper Functions - Acquisition Dataframes 
# -----------------------------------------------------------------------------
def build_datasets(config_df, base_dir):
    """
    Constructs dataset and background DataFrames by iterating over the config_df.
    Uses each row to construct two file paths (for two microphones).
    """
    dataset_records = []
    background_records = []
    
    # Only process rows where 'ID acq' is non-null
    config_df = config_df[config_df['ID acq'].notna()]
    
    for _, row in config_df.iterrows():
        acq_id = row['ID acq']
        file_folder = os.path.join(base_dir, str(acq_id), '_Exported')
        sensor_paths = {s_name : os.path.join(file_folder, f'{s_name}.parquet') for s_name in ['imp23absu_mic', 'imp34dt05_mic', 'ism330dhcx_acc', 'ism330dhcx_gyro']}

        #mic1_file = os.path.join(file_folder, 'imp23absu_mic.wav')
        #mic2_file = os.path.join(file_folder, 'imp34dt05_mic.wav')
        
        is_background = (not pd.isna(row.get('Bkg noise'))) and (str(row.get('Bkg noise')).strip() != '\\')
        
        if is_background:
            # for mic_file in [mic1_file, mic2_file]:
            bkg_id = str(row.get('Bkg noise')).strip()
            domain_flag = BACKGROUND_DOMAIN_MAPPING.get(bkg_id, 'unknown')
            background_records.append({
                'ID acq': acq_id,
                'file path': sensor_paths['imp34dt05_mic'],
                'pos': row.get('pos'),
                'background ID': bkg_id,
                'domain_flag': domain_flag                    
            })
        else:
            
            domain_flag = 'source' if row.get('grinder precision (um)') in SOURCE_DOMAIN_GRINDS else 'target'
            tmp_dict = {
                'ID acq': acq_id,
                'Model': row.get('Model'),
                'Type': row.get('Type'),
                'position': row.get('pos'),
                'grind size': row.get('grinder precision (um)'),
                'anomaly type': row.get('Anomaly type'),
                'source_target_flag': domain_flag # domain flag for segments.
            }
            for sensor_name, sensor_path in sensor_paths.items():
                tmp_dict[sensor_name] = sensor_path
            dataset_records.append(tmp_dict)
    
    dataset_df = pd.DataFrame(dataset_records)
    background_df = pd.DataFrame(background_records)
    return dataset_df, background_df

In [6]:
 # Step 2: Build the primary dataset and background DataFrames.
dataset_df, background_df = build_datasets(config_df, BASE_DIR)
dataset_df.to_csv(os.path.join(BASE_DIR, 'dataset.csv'), index=False)
background_df.to_csv(os.path.join(BASE_DIR, 'background.csv'), index=False)
logging.info("Datasets built and saved.")

dataset_df

INFO: Datasets built and saved.


,ID acq,Model,Type,position,grind size,anomaly type,source_target_flag,imp23absu_mic,imp34dt05_mic,ism330dhcx_acc,ism330dhcx_gyro
0,20250404_15_43_05,White,Normal,A,0,NaN,source,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250404...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250404...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250404...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250404...
1,20250404_15_41_56,White,Normal,A,0,NaN,source,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250404...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250404...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250404...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250404...
2,20250408_16_29_45,White,Normal,A,0,NaN,source,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250408...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250408...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250408...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250408...
3,20250408_16_28_39,White,Normal,A,0,NaN,source,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250408...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250408...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250408...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250408...
4,20250403_14_52_31,White,Normal,A,0,NaN,source,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250403...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250403...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250403...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250403...
...,...,...,...,...,...,...,...,...,...,...,...
103,20250411_15_31_37,White,Anomaly,B,300,loose_screw,target,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250411...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250411...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250411...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250411...
104,20250411_15_21_57,White,Anomaly,C,0,loose_screw,source,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250411...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250411...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250411...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250411...
105,20250411_15_17_32,White,Anomaly,C,100,loose_screw,source,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250411...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250411...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250411...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250411...
106,20250411_15_13_18,White,Anomaly,C,200,loose_screw,source,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250411...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250411...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250411...,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250411...


In [7]:
background_df

,ID acq,file path,pos,background ID,domain_flag
0,20250403_16_20_54,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250403...,A,A,source
1,20250403_16_34_37,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250403...,A,A,source
2,20250403_16_38_15,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250403...,B,A,source
3,20250404_09_38_49,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250404...,C,A,source
4,20250404_11_49_23,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250404...,A,B,source
5,20250404_12_20_16,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250404...,B,B,source
6,20250404_14_29_48,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250404...,C,B,source
7,20250404_15_54_56,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250404...,A,C,source
8,20250404_15_23_23,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250404...,B,C,source
9,20250404_16_11_21,D:\IMADS\CoffeeGrinder\STWINonGrinder\20250404...,C,C,source


<h2>Step 3: Export audio segments into files and build segments metadata.</h2>

In [ ]:
# -----------------------------------------------------------------------------
# Helper Functions - Extract Segments 
# -----------------------------------------------------------------------------
def extract_mic_type(file_path):
    """
    Extracts microphone type from a file path. Assumes the file name is of the form: imp23absu_mic.wav
    """
    basename = os.path.basename(file_path)
    name_without_ext = os.path.splitext(basename)[0]
    return name_without_ext
 
def export_segments(df, segments_dir, duration_in_seconds, sensors = ['imp34dt05_mic', 'ism330dhcx_acc', 'ism330dhcx_gyro']):
    """
    Reads each audio file from df, segments the file into fixed duration chunks,
    writes the segments to the output directory, and returns a DataFrame with metadata.
    """
    os.makedirs(segments_dir, exist_ok=True)
    rows_dfs = []
    # read parquet column
    sensors_columns = {
        'imp34dt05_mic': [
            'MIC [Waveform]'
            ],
        'ism330dhcx_acc': [
            'A_x [g]',
            'A_y [g]',
            'A_z [g]'
            ],
        'ism330dhcx_gyro': [
            'G_x [dps]',
            'G_y [dps]',
            'G_z [dps]'
            ]
        }
    
    for _, row in df.iterrows():

        acq_metadata_dict = {
            'ID acq': row.get('ID acq'),
            'domain_shift_grind_size': row.get('grind size'),
            'anomaly type': row.get('anomaly type'),
            'source_target_flag': row.get('source_target_flag'),
            'anomaly_label': 'normal' if row.get('Type') == 'Normal' else 'anomaly',
        }

        start_time = min([pd.read_parquet(row[sensor], columns=['Time']).iloc[0,0] for sensor in sensors])
        end_time = max([pd.read_parquet(row[sensor], columns=['Time']).iloc[-1,0] for sensor in sensors])

        segments_starts = np.arange(start_time, end_time, duration_in_seconds) # TO CHECK

        for i in range(len(segments_starts)-1):
            # create dataframe row for each segment from metadata_dict
            new_row = pd.DataFrame(acq_metadata_dict, index=[0])
            this_segment_filter= [('Time', '>=', segments_starts[i]), ('Time', '<', segments_starts[i+1])]
            for sensor in sensors:
                segment = pd.read_parquet(row[sensor], filters=this_segment_filter)
                filename = f'{sensor}_{row.get("ID acq")}_{i}.parquet'
                file_out_path = os.path.join(segments_dir, filename)

                try:
                    # write as a parquet file
                    segment_df = pd.DataFrame(segment)
                    segment_df.to_parquet(file_out_path, index=False)
                except Exception as e:
                    logging.error(f"Error writing segment {file_out_path}: {e}")
                    continue

                new_row[sensor] = file_out_path
            
            rows_dfs.append(new_row)
 
    df = pd.concat(rows_dfs, ignore_index=True, axis=0)
    return df

In [17]:
# Step 3: Export audio segments into files and build segments metadata.
sensors =  ['imp34dt05_mic', 'ism330dhcx_acc', 'ism330dhcx_gyro']
segments_df = export_segments(dataset_df, SEGMENTS_DIR, DURATION_IN_SECONDS, sensors = sensors)
segments_df.to_csv(os.path.join(BASE_DIR, 'segments.csv'), index=False)
logging.info("Audio segments exported and metadata saved.")
segments_df

INFO: Audio segments exported and metadata saved.


,ID acq,domain_shift_grind_size,anomaly type,source_target_flag,anomaly_label,imp34dt05_mic,ism330dhcx_acc,ism330dhcx_gyro
0,20250404_15_43_05,0,NaN,source,normal,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...
1,20250404_15_43_05,0,NaN,source,normal,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...
2,20250404_15_43_05,0,NaN,source,normal,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...
3,20250404_15_43_05,0,NaN,source,normal,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...
4,20250404_15_43_05,0,NaN,source,normal,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...
...,...,...,...,...,...,...,...,...
1292,20250411_15_15_27,300,loose_screw,target,anomaly,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...
1293,20250411_15_15_27,300,loose_screw,target,anomaly,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...
1294,20250411_15_15_27,300,loose_screw,target,anomaly,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...
1295,20250411_15_15_27,300,loose_screw,target,anomaly,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...,D:\IMADS\CoffeeGrinder\STWINonGrinder\segments...


<h2>Step 4: Mix background noise into segments.</h2>

In [2]:
segments_df = pd.read_csv(os.path.join(BASE_DIR, 'segments.csv'))
background_df = pd.read_csv(os.path.join(BASE_DIR, 'background.csv'))
dataset_df = pd.read_csv(os.path.join(BASE_DIR, 'dataset.csv'))

In [76]:
# -----------------------------------------------------------------------------
# Helper Functions - Mix Background Noise
# -----------------------------------------------------------------------------
from tqdm import tqdm

def mix_background_noise(segments_df, background_df, segments_dir, base_dir, duration_in_seconds, sampling_rate, target_snr_db):
    """
    Mixes background noise into each target segment.
    Uses caching to avoid recalculating durations and maintains a read head per background file.
    Updates the segments_df with the applied background 'background ID'.
    """
    bkg_info_cache = {}
    bkg_read_heads = {}  # Structure: { mic_type: {acq_id: current_sample} }

    mic_types = background_df['file path'].apply(extract_mic_type).unique()
    for mic in mic_types:
        acq_ids = background_df['ID acq'].unique()
        bkg_read_heads[mic] = {acq: 0 for acq in acq_ids}

    #shuffle segments_df to randomize processing order
    segments_df = segments_df.sample(frac=1, random_state=42).reset_index(drop=True)

    for i, row in tqdm(segments_df.iterrows(), desc= 'Processing segments:', total=len(segments_df)):
        # segment_pos = row.get('position')
        segment_mic_type = 'imp34dt05_mic' #row.get('mic_type')
        segment_domain = row.get('source_target_flag')
        
        # Filter background rows matching both position and mic type.
        bkg_candidates = background_df[
            # (background_df['pos'] == segment_pos) &
            # (background_df['file path'].str.contains(segment_mic_type)) &
            background_df['domain_flag'] == segment_domain
        ]
        if bkg_candidates.empty:
            logging.warning(f"No background candidates for segment index {i}.")
            continue
        
        # get the duration in seconds for each background file
        # and check if it can be used for the current segment 
        # and has enough duration to mix with the target segment.
        legal_bkg_records = []
        for acq_id in bkg_candidates['ID acq'].unique():
            bkg_row = bkg_candidates[bkg_candidates['ID acq'] == acq_id].iloc[0] # convert to Series
            bkg_file = bkg_row['file path']
            if bkg_file not in bkg_info_cache:
                try:
                    bkg_df = pd.read_parquet(bkg_file, columns=['Time'])
                    offset = bkg_df.iloc[0].item()
                    bkg_info_cache[bkg_file] = {
                        'duration':bkg_df.iloc[-1].item() - offset,
                        'start_offset': offset
                    }

                    # initialize read head to each background file start offset
                    bkg_read_heads[segment_mic_type][acq_id]= offset

                except Exception as e:
                    logging.error(f"Failed to get duration for {bkg_file}: {e}")
                    continue

            current_read_head = bkg_read_heads.get(segment_mic_type, {}).get(acq_id, 0)
            if current_read_head != -1 and ((current_read_head + duration_in_seconds) <= (bkg_info_cache[bkg_file]['start_offset'] + bkg_info_cache[bkg_file]['duration'])):
                legal_bkg_records.append(bkg_row)
        
        if not legal_bkg_records:
            logging.warning(f"No legal background file available for segment index {i}.")
            continue

        legal_bkg_df = pd.DataFrame(legal_bkg_records)
        shuffled_legal = legal_bkg_df.sample(frac=1)  # randomize order (without fixed seed)
        bkg_row = shuffled_legal.iloc[0] # choose the first one after shuffling
        bkg_acq_id = bkg_row['ID acq']
        bkg_file = bkg_row['file path']
        bkg_id = bkg_row['background ID']

        current_read_head = bkg_read_heads.get(segment_mic_type, {}).get(bkg_acq_id, 0)
        start_time = current_read_head 
        end_time = start_time + duration_in_seconds
        
        try:
            this_chunk_filter= [('Time', '>=', start_time ), ('Time', '<', end_time)]
            bkg_audio = pd.read_parquet(bkg_file, columns=['MIC [Waveform]'], filters=this_chunk_filter)
        except Exception as e:
            logging.error(f"Error loading background audio chunk from {bkg_file}: {e}")
            continue
        
        # Update read head
        if (end_time + duration_in_seconds) >= (bkg_info_cache[bkg_file]['start_offset'] + bkg_info_cache[bkg_file]['duration']):
            bkg_read_heads[segment_mic_type][bkg_acq_id] = -1
        else:
            bkg_read_heads[segment_mic_type][bkg_acq_id] = end_time
        
        # Load target segment audio
        target_file_path = os.path.join(base_dir, row['imp34dt05_mic'])
        try:
            target_audio = pd.read_parquet(target_file_path, columns=['MIC [Waveform]'])
        except Exception as e:
            logging.error(f"Error loading target audio {target_file_path}: {e}")
            continue
        
        # Compute scaling factor using SNR
        signal_power = np.mean(target_audio ** 2)
        noise_power = np.mean(bkg_audio ** 2)
        target_noise_power = signal_power / (10 ** (target_snr_db / 10))
        scaling_factor = np.sqrt(target_noise_power / (noise_power + 1e-10))
        scaled_bkg = bkg_audio * scaling_factor
        mixed_audio = target_audio + scaled_bkg
        
        # Update the DataFrame with the applied background noise ID
        segments_df.at[i, 'background ID'] = bkg_id
        
        # Save mixed audio: update the file name to include the background ID.
        old_filepath = row['imp34dt05_mic']
        mixed_filename = old_filepath.replace('.parquet', f'_bkg_{bkg_id}.parquet')
        mixed_filepath = os.path.join(segments_dir, os.path.basename(mixed_filename))
        try:
            # sf.write(mixed_filepath, mixed_audio, sampling_rate)
            mixed_audio.to_parquet(mixed_filepath, index=False)
        except Exception as e:
            logging.error(f"Error writing mixed audio to {mixed_filepath}: {e}")
            continue

        segments_df.at[i, 'imp34dt05_mic'] = mixed_filepath  # update relative path
            
    return segments_df

In [77]:
# Step 4: Mix background noise into segments.

segments_df = mix_background_noise(segments_df, background_df, SEGMENTS_DIR, BASE_DIR, DURATION_IN_SECONDS, SAMPLING_RATE_MIC, TARGET_SNR_DB)
segments_df.to_csv(os.path.join(BASE_DIR, 'segments_with_bkg.csv'), index=False)

Processing segments:: 100%|██████████| 1297/1297 [04:15<00:00,  5.08it/s]


In [87]:
segments_df['segment_ID']=segments_df['imp34dt05_mic'].apply(lambda x: x.split('\\')[-1].split('_bkg_')[0].replace('imp34dt05_mic_',''))
# create a column 'segment_id' for each row of segments_df that takes the initial part of the filename (before '_bkg_' from 'imp34dt05_mic' value, and removes 'imp34dt05_mic_' 
segments_df.to_csv(os.path.join('.', 'segments_with_bkg.csv'), index=False)

<h2>Step 5: Rename anomaly files (change 'train' to 'test') and update metadata.</h2>

In [ ]:
def rename_anomaly_files(segments_df, base_dir):
    """
    For anomaly segments, renames the file path by replacing 'train' with 'test'
    and updates the DataFrame accordingly.
    """
    for i, row in segments_df.iterrows():
        if 'anomaly' in row.get('file path', ''):
            old_rel = row['file path']
            new_rel = old_rel.replace('train', 'test')
            old_abs = os.path.join(base_dir, old_rel)
            new_abs = os.path.join(base_dir, new_rel)
            try:
                os.rename(old_abs, new_abs)
            except Exception as e:
                logging.error(f"Error renaming {old_abs} to {new_abs}: {e}")
                continue
            segments_df.at[i, 'file path'] = new_rel
            segments_df.at[i, 'train_test_flag'] = 'test'
    return segments_df

In [ ]:
# Step 5: Rename anomaly files (change 'train' to 'test') and update metadata.
segments_df = rename_anomaly_files(segments_df, BASE_DIR)

<h2>Step 6: Create  new column for stratification labels.</h2>

In [ ]:
# Step 6: Convert DataFrame columns to string and build single-stage target label.
segments_df = segments_df.astype(str)
segments_df['target_label'] = (
    segments_df['source_target_flag'] + '_' +
    segments_df['grind size'] + '_' +
    segments_df['position'] + '_' +
    segments_df['mic_type'] + '_' +
    segments_df['background ID']
)

segments_df

<h2>Step 7: Split into train and test sets using stratification based on target_label.</h2>

In [ ]:
def stratified_sample(df, target_column, target_samples):
    """
    Performs stratified sampling on the DataFrame to achieve a total of target_samples overall.
    """
    class_counts = df[target_column].value_counts()
    class_ratios = class_counts / class_counts.sum()
    samples_per_class = (class_ratios * target_samples).round().astype(int)
    
    sampled_dfs = []
    for cls in class_counts.index:
        available = class_counts.loc[cls]
        n_samples = min(samples_per_class[cls], available)
        try:
            sampled = df[df[target_column] == cls].sample(n=n_samples, random_state=42)
            sampled_dfs.append(sampled)
        except Exception as e:
            logging.error(f"Error sampling class {cls}: {e}")
    return pd.concat(sampled_dfs) if sampled_dfs else pd.DataFrame()

def split_train_test(segments_df, test_size=30):
    """
    Splits the segments DataFrame into train and test sets.
    Stratification is based on a single-stage target_label that now includes
    background noise information.
    """

    segments_normal = segments_df[segments_df['normal_anomaly_flag'] == 'normal']
    segments_anomaly = segments_df[segments_df['normal_anomaly_flag'] == 'anomaly']
    print(f"Normal segments: {len(segments_normal)}, Anomaly segments: {len(segments_anomaly)}")
    
    normal_source = segments_normal[segments_normal['source_target_flag'] == 'source']
    normal_target = segments_normal[segments_normal['source_target_flag'] == 'target']
    print(f"Normal source segments: {len(normal_source)}, Normal target segments: {len(normal_target)}")
    
    if not normal_source.empty:
        train_ns, test_ns = train_test_split(
            normal_source,
            test_size=test_size,
            stratify=normal_source['target_label'],
            random_state=42
        )
    else:
        train_ns, test_ns = pd.DataFrame(), pd.DataFrame()
    
    if not normal_target.empty:
        train_nt, test_nt = train_test_split(
            normal_target,
            test_size=test_size,
            stratify=normal_target['target_label'],
            random_state=42
        )
    else:
        train_nt, test_nt = pd.DataFrame(), pd.DataFrame()
    
    test_anomaly = stratified_sample(segments_anomaly, 'target_label', target_samples=100)
    
    train_df = pd.concat([train_ns, train_nt], ignore_index=True)
    test_df = pd.concat([test_ns, test_nt, test_anomaly], ignore_index=True)
    
    return train_df, test_df

In [ ]:
# Step 7: Split into train and test sets using stratification based on target_label.
train_df, test_df = split_train_test(segments_df, test_size=90)
logging.info("Train/test splitting completed.")

# show the number of samples in each set
print(f"Train samples: {len(train_df)}, Test samples: {len(test_df)}")
# show the number of samples in each class
print("Train class distribution:")
print(train_df['target_label'].value_counts())
print("Test class distribution:")
print(test_df['target_label'].value_counts())


<h2>Step 8: Copy (and rename) files into train and test directories.</h2>

In [ ]:
def copy_files_to_dirs(train_df, test_df, base_dir, segments_dir):
    """
    Copies segment files into separate 'train' and 'test' directories.
    Files in the test set are renamed from 'train' to 'test' if needed.
    """
    train_dir = os.path.join(segments_dir, 'train')
    test_dir = os.path.join(segments_dir, 'test')
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    
    for file_rel in train_df['file path']:
        src = os.path.join(base_dir, file_rel)
        dst = os.path.join(train_dir, os.path.basename(file_rel))
        try:
            shutil.copy(src, dst)
        except Exception as e:
            logging.error(f"Error copying {src} to {dst}: {e}")
    
    for i, row in test_df.iterrows():
        old_rel = row['file path']
        new_rel = old_rel.replace('train', 'test')
        old_abs = os.path.join(base_dir, old_rel)
        new_abs = os.path.join(base_dir, new_rel)
        try:
            os.rename(old_abs, new_abs)
        except Exception as e:
            logging.error(f"Error renaming {old_abs} to {new_abs}: {e}")
            continue
        test_df.at[i, 'file path'] = new_rel
        test_df.at[i, 'train_test_flag'] = 'test'
        try:
            shutil.copy(new_abs, test_dir)
        except Exception as e:
            logging.error(f"Error copying {new_abs} to {test_dir}: {e}")
    
    return train_df, test_df

In [ ]:
# Step 8: Copy (and rename) files into train and test directories.
train_df, test_df = copy_files_to_dirs(train_df, test_df, BASE_DIR, SEGMENTS_DIR)

final_segments_csv = os.path.join(BASE_DIR, 'segments_final.csv')
segments_df.to_csv(final_segments_csv, index=False)

test_count = segments_df['file path'].str.contains('test').sum()
logging.info(f"Number of test files in segments metadata: {test_count}")